# Pretraining Mini: Masked Language Modeling lalu Fine-Tuning

**Buku:** *Deep Learning dengan Python* — Bab 9, Pretraining Skala Besar dan Model Fondasi

Resep model fondasi dalam skala kecil. Encoder Transformer dilatih dulu menebak kata yang disembunyikan pada 75 ribu ulasan **tanpa label**. Setelah itu model di-*fine-tune* dengan hanya 2.000 ulasan berlabel dan dibandingkan dengan model yang dilatih dari nol.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai backend **TensorFlow** (dibutuhkan oleh `tf.data`/`tf.GradientTape`/ekspor).

**Perkiraan waktu (GPU T4):** ±30 menit

**Isi notebook:**
- Tokenisasi dan penambahan token khusus `[MASK]`.
- Masking ala BERT: 15% token dipilih; 80% → `[MASK]`, 10% → kata acak, 10% tetap.
- *Dynamic masking* (masker baru setiap epoch, seperti RoBERTa).
- Transfer bobot encoder ke pengklasifikasi sentimen, lalu dibandingkan dengan pelatihan dari nol.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf

## 1. Data: ulasan berlabel dan tidak berlabel

In [ ]:
import tarfile, pathlib, numpy as np

def siapkan_aclimdb(folder="data"):
    """Siapkan data/aclImdb/{train,test}/{neg,pos,unsup}/*.txt.
    Sumber utama: server Stanford. Bila tidak dapat dihubungi, unduh salinan
    resmi yang sama dari Hugging Face (stanfordnlp/imdb)."""
    akar = pathlib.Path(folder) / "aclImdb"
    if (akar / "train" / "pos").exists():
        return
    try:
        jalur = keras.utils.get_file(
            "aclImdb_v1.tar.gz",
            origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz")
        with tarfile.open(jalur) as berkas_tar:
            berkas_tar.extractall(folder)
    except Exception as galat:
        print("Server Stanford tidak dapat dihubungi:", galat)
        print("Memakai salinan Hugging Face (stanfordnlp/imdb) ...")
        from datasets import load_dataset
        ds = load_dataset("stanfordnlp/imdb")
        for bagian, nama in [("train", "train"), ("test", "test"), ("unsupervised", "train")]:
            for i, contoh in enumerate(ds[bagian]):
                jenis = "unsup" if bagian == "unsupervised" else ("pos" if contoh["label"] == 1 else "neg")
                tujuan = akar / nama / jenis
                tujuan.mkdir(parents=True, exist_ok=True)
                (tujuan / f"{bagian}_{i}.txt").write_text(contoh["text"], encoding="utf-8")

siapkan_aclimdb()

def baca_folder(folder, jenis=(("neg", 0), ("pos", 1))):
    teks, label = [], []
    for nama, nilai in jenis:
        for p in sorted((pathlib.Path(folder) / nama).glob("*.txt")):
            teks.append(p.read_text(encoding="utf-8"))
            label.append(nilai)
    return teks, np.array(label)

rng = np.random.default_rng(42)
def acak(teks, label):
    urutan = rng.permutation(len(teks))
    return [teks[i] for i in urutan], label[urutan]

teks_latih, y_latih = acak(*baca_folder("data/aclImdb/train"))
teks_uji, y_uji = acak(*baca_folder("data/aclImdb/test"))
print(len(teks_latih), "ulasan latih |", len(teks_uji), "ulasan uji")

teks_unsup, _ = baca_folder("data/aclImdb/train", jenis=(("unsup", -1),))
teks_pretrain = teks_latih + teks_unsup          # data uji TIDAK dipakai saat pretraining
print(len(teks_pretrain), "ulasan untuk pretraining")

## 2. Tokenisasi
Token `[MASK]` diberi indeks tambahan `V` di luar kosakata, sehingga embedding berukuran `V + 1`.

In [ ]:
from keras import layers

V, L = 15000, 128

def standarisasi(s):
    s = tf.strings.lower(s)
    s = tf.strings.regex_replace(s, "<br />", " ")
    return tf.strings.regex_replace(s, "[^a-z0-9' ]", " ")

vek = layers.TextVectorization(max_tokens=V, output_sequence_length=L, standardize=standarisasi)
vek.adapt(tf.data.Dataset.from_tensor_slices(teks_pretrain).batch(1024))
kosakata = vek.get_vocabulary()
ID_MASK = V

def ke_id(teks, batch=4096):
    return np.concatenate([keras.ops.convert_to_numpy(vek(teks[i:i + batch]))
                           for i in range(0, len(teks), batch)]).astype("int32")

id_pretrain = ke_id(teks_pretrain)
print(id_pretrain.shape, "| contoh:", [kosakata[i] for i in id_pretrain[0, :12]])

## 3. Masking ala BERT

In [ ]:
def buat_mlm(id_token, rng, p=0.15):
    x = id_token.copy()
    pilih = (rng.random(x.shape) < p) & (x != 0)          # jangan pilih padding
    u = rng.random(x.shape)
    x[pilih & (u < 0.8)] = ID_MASK                         # 80% -> [MASK]
    ganti = pilih & (u >= 0.8) & (u < 0.9)                 # 10% -> kata acak
    x[ganti] = rng.integers(2, V, size=int(ganti.sum()))
    # 10% sisanya dibiarkan, tetapi tetap harus ditebak
    return x, id_token, pilih.astype("float32")            # bobot sampel: hanya posisi terpilih yang dihitung

x_mlm, y_mlm, w_mlm = buat_mlm(id_pretrain[:1], np.random.default_rng(0))
print("Asli   :", " ".join(kosakata[i] for i in y_mlm[0, :20]))
print("Masukan:", " ".join("[MASK]" if i == ID_MASK else kosakata[i] for i in x_mlm[0, :20]))

## 4. Model MLM

In [ ]:
from keras import layers, ops

class EmbeddingPosisi(layers.Layer):
    """Embedding token + embedding posisi yang dipelajari."""
    def __init__(self, panjang, kosakata, dim, **kwargs):
        super().__init__(**kwargs)
        self.token = layers.Embedding(kosakata, dim)
        self.posisi = layers.Embedding(panjang, dim)
        self.cfg = dict(panjang=panjang, kosakata=kosakata, dim=dim)

    def call(self, x):
        pos = ops.arange(0, ops.shape(x)[-1])
        return self.token(x) + self.posisi(pos)

    def get_config(self):
        return {**super().get_config(), **self.cfg}


class EncoderTransformer(layers.Layer):
    """Satu blok encoder: self-attention + FFN, masing-masing dengan residual dan LayerNorm."""
    def __init__(self, dim, dim_ff, n_head, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.atensi = layers.MultiHeadAttention(num_heads=n_head, key_dim=dim // n_head)
        self.ffn = keras.Sequential([layers.Dense(dim_ff, activation="gelu"), layers.Dense(dim)])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.cfg = dict(dim=dim, dim_ff=dim_ff, n_head=n_head, dropout=dropout)

    def call(self, x, token=None, training=False):
        mask = None
        if token is not None:                                  # abaikan posisi padding (token 0)
            mask = ops.expand_dims(ops.not_equal(token, 0), 1)
        a = self.atensi(x, x, attention_mask=mask, training=training)
        x = self.norm1(x + self.drop1(a, training=training))
        return self.norm2(x + self.drop2(self.ffn(x), training=training))

    def get_config(self):
        return {**super().get_config(), **self.cfg}

In [ ]:
DIM, DIM_FF, N_HEAD, N_BLOK = 128, 256, 4, 2

def bangun_encoder():
    masukan = keras.Input(shape=(L,), dtype="int32")
    x = EmbeddingPosisi(L, V + 1, DIM, name="embedding")(masukan)
    for i in range(N_BLOK):
        x = EncoderTransformer(DIM, DIM_FF, N_HEAD, name=f"encoder_{i}")(x, masukan)
    return keras.Model(masukan, x, name="encoder")

encoder_mlm = bangun_encoder()
mlm = keras.Model(encoder_mlm.input,
                  layers.Dense(V, activation="softmax", name="kepala_mlm")(encoder_mlm.output))
mlm.compile(optimizer=keras.optimizers.Adam(5e-4),
            loss="sparse_categorical_crossentropy", weighted_metrics=["accuracy"])
mlm.summary()

## 5. Pretraining dengan *dynamic masking*

In [ ]:
EPOCH_PRETRAIN = 10
rng_mask = np.random.default_rng(1)
for epoch in range(1, EPOCH_PRETRAIN + 1):
    print(f"Epoch pretraining {epoch}/{EPOCH_PRETRAIN}")
    x, y, w = buat_mlm(id_pretrain, rng_mask)              # masker baru setiap epoch
    mlm.fit(x, y, sample_weight=w, batch_size=32, epochs=1)
encoder_mlm.save_weights("encoder_mlm.weights.h5")

### Uji tebak kata
Tulis `___` di posisi kata yang ingin ditebak model.

In [ ]:
indeks_kata = {k: i for i, k in enumerate(kosakata)}

def tebak(kalimat, k=5):
    token = keras.ops.convert_to_numpy(vek([kalimat.replace("___", "qqmaskqq")]))[0]
    posisi = int(np.where(token == indeks_kata.get("qqmaskqq", 1))[0][0])
    token[posisi] = ID_MASK
    prob = keras.ops.convert_to_numpy(mlm(token[None], training=False))[0, posisi]
    teratas = np.argsort(prob)[::-1][:k]
    print(f"{kalimat}\n   -> " + ", ".join(f"{kosakata[i]} ({prob[i]:.2f})" for i in teratas))

tebak("this movie was ___ good")
tebak("the acting was terrible and the plot was ___")
tebak("one of the best ___ i have ever seen")

## 6. Fine-tuning dengan sedikit label
Skenario yang sama, dua kali: encoder **pralatih** dan encoder **acak** (dari nol).

In [ ]:
N_BERLABEL = 2000
id_latih, id_uji = ke_id(teks_latih), ke_id(teks_uji)
x_kecil, y_kecil = id_latih[:N_BERLABEL], y_latih[:N_BERLABEL]
x_val, y_val = id_latih[-5000:], y_latih[-5000:]

def pengklasifikasi(pralatih):
    enc = bangun_encoder()
    if pralatih:
        enc.load_weights("encoder_mlm.weights.h5")
    x = layers.GlobalAveragePooling1D()(enc.output)
    x = layers.Dropout(0.3)(x)
    model = keras.Model(enc.input, layers.Dense(1, activation="sigmoid")(x))
    model.compile(optimizer=keras.optimizers.Adam(2e-4), loss="binary_crossentropy", metrics=["accuracy"])
    return model

hasil = {}
for pralatih in [True, False]:
    nama = "pralatih MLM" if pralatih else "dari nol"
    model = pengklasifikasi(pralatih)
    model.fit(x_kecil, y_kecil, batch_size=32, epochs=15, validation_data=(x_val, y_val),
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                                       restore_best_weights=True)], verbose=0)
    hasil[nama] = model.evaluate(id_uji, y_uji, verbose=0)[1]
    print(f"{nama:13s}: akurasi uji {hasil[nama]:.2%}")

Selisih akurasi tersebut berasal dari pengetahuan bahasa yang diperoleh **tanpa label sama sekali**. Model fondasi seperti BERT dan GPT memakai resep yang sama, hanya dengan data dan model jutaan kali lebih besar.

**Eksperimen:** ubah `N_BERLABEL` menjadi 200 atau 20.000. Kapan keunggulan pretraining menyusut?